# ntl-l24-glm9-s2

Stage 2 of the two-stage LoRA24 member glm9v1 (stage 1: ntl-l24-glm9-s1b): two epochs on the 2,699 official rows, once per held-out fold of the split inferred from nlm-v27's own fold adapters (out-of-fold predictions) and once on all rows, in the layout of Dhruv Tibarewal's nlm-l24-adapters-v5. Training kernel, not a submission.

In [ ]:
%%writefile /kaggle/working/l24_worker.py
"""Two-stage LoRA24 member trainer for Name That LLM, saved in the format nlm-v27's infer.py loads.

Lucas Boesen's public LoRA24 recipe (kernel name-that-llm-train-lora24-s7) with Dhruv Tibarewal's two stages (nlm-v27:
stage 1 on his 120k public pool with class weights, stage 2 on the 2,699 official rows): the base truncated to --keep
layers in float16 (frozen), LoRA r16 / alpha 32 / dropout .05 on every linear layer of the decoder (fp32), masked mean
pool (first token dropped) of 8 depths with one LayerNorm each, concatenated, into a plain and a prompt-centred linear
head; micro-batches of same-prompt triplets; loss CE(plain(z)) + CE(centred(z - triplet mean)); AdamW (adapter and
head learning rates), weight decay .01, warmup then cosine, clip 1.0, fp16 autocast with a GradScaler, gradient
checkpointing. A multimodal checkpoint (Ministral 3) keeps its text path only; infer.py loads the same AutoModel.

    python l24_worker.py --stage 0 ...                         fp16 health, memory and speed check (one GPU)
    torchrun --nproc_per_node 2 l24_worker.py --stage 1 ...    pool stage, data-parallel over both GPUs, time-boxed
    python l24_worker.py --stage 2 --jobs 0,2,4 ...            official rows, one job per held-out fold (or 'full')

Output of stage 1 and of each stage-2 job: adapter_config.json + adapter_model.safetensors (peft), heads.pt (norms.i,
plain, centered; rows in --labels order) and meta.json; stage-2 fold jobs also write valid.npz (held-out log-probs).
"""
import argparse
import contextlib
import importlib.util as _iu
import json
import math
import os
import time

_fs = _iu.find_spec   # Kaggle ships a torchao that some peft versions reject; it is unused here
_iu.find_spec = lambda name, *a, **k: None if name.split('.')[0] == 'torchao' else _fs(name, *a, **k)
import numpy as np
import pandas as pd
import torch
import torch.distributed
import torch.nn.functional as F
from torch import nn

T0 = time.time()
TARGET = r'(.*\.)?layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|gate_up_proj|down_proj))'


def parse():
    p = argparse.ArgumentParser()
    p.add_argument('--stage', type=int, required=True)
    p.add_argument('--base', required=True); p.add_argument('--out', required=True)
    p.add_argument('--labels', required=True, help='json list: head row order')
    p.add_argument('--keep', type=int, default=24); p.add_argument('--layers', default='2,4,6,8,12,16,20,24')
    p.add_argument('--max-len', type=int, default=384)
    p.add_argument('--r', type=int, default=16); p.add_argument('--alpha', type=int, default=32)
    p.add_argument('--ldrop', type=float, default=0.05)
    p.add_argument('--lr', type=float, default=1e-4); p.add_argument('--head-lr', type=float, default=1e-3)
    p.add_argument('--wd', type=float, default=0.01); p.add_argument('--warmup', type=float, default=0.03)
    p.add_argument('--epochs', type=float, default=2.0)
    p.add_argument('--tri', type=int, default=2, help='triplets per micro-batch')
    p.add_argument('--accum', type=int, default=2, help='micro-batches per optimizer step and GPU')
    p.add_argument('--hours', type=float, default=9.0, help='stage 1: training time budget')
    p.add_argument('--ckpt-min', type=float, default=60.0)
    p.add_argument('--pool', default=''); p.add_argument('--train', default='')
    p.add_argument('--init', default='', help='stage 1: an earlier stage-1 output to continue; stage 2: the stage-1 output')
    p.add_argument('--jobs', default='', help="stage 2: comma list of held-out folds and/or 'full'")
    p.add_argument('--class-weights', action='store_true')
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--check-rows', type=int, default=48, help='stage 0: rows per check')
    a = p.parse_args()
    a.layers = [int(x) for x in a.layers.split(',')]; a.labels = json.loads(a.labels)
    return a


RANK, WORLD = int(os.environ.get('RANK', 0)), int(os.environ.get('WORLD_SIZE', 1))
LOCAL = int(os.environ.get('LOCAL_RANK', 0))
DEV = torch.device(f'cuda:{LOCAL}') if torch.cuda.is_available() else torch.device('cpu')
CUDA = DEV.type == 'cuda'


def log(*x):
    if RANK == 0:
        print(f'[{(time.time() - T0) / 60:7.1f} min]', *x, flush=True)


def autocast():
    return torch.autocast('cuda', dtype=torch.float16) if CUDA else contextlib.nullcontext()


def load_base(a):
    from transformers import AutoConfig, AutoModel, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(a.base)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    cfg = AutoConfig.from_pretrained(a.base); tc = cfg.text_config if hasattr(cfg, 'text_config') else cfg
    tc.num_hidden_layers = a.keep; tc.use_cache = False
    if getattr(tc, 'layer_types', None) is not None:
        tc.layer_types = tc.layer_types[:a.keep]
    m = AutoModel.from_pretrained(a.base, config=cfg, dtype=torch.float16 if CUDA else torch.float32)
    for name in ('vision_tower', 'multi_modal_projector'):   # text only: the image path never runs
        if getattr(m, name, None) is not None:
            setattr(m, name, None)
    m.to(DEV)
    m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    m.enable_input_require_grads()
    return tok, m, tc.hidden_size


def add_lora(m, a, init=''):
    from peft import LoraConfig, PeftModel, get_peft_model
    if init:
        model = PeftModel.from_pretrained(m, init, is_trainable=True)
    else:
        model = get_peft_model(m, LoraConfig(r=a.r, lora_alpha=a.alpha, lora_dropout=a.ldrop, bias='none',
                                             target_modules=TARGET))
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert n > 0, 'no LoRA parameters: target modules did not match'
    return model, n


class Net(nn.Module):
    """LoRA decoder + one LayerNorm per pooled depth + plain and centred heads; forward returns the loss."""

    def __init__(self, model, hid, layers, k):
        super().__init__()
        self.model, self.layers = model, layers
        self.norms = nn.ModuleList([nn.LayerNorm(hid) for _ in layers])
        self.plain = nn.Linear(hid * len(layers), k)
        self.centered = nn.Linear(hid * len(layers), k)
        self.drop = nn.Dropout(0.1)

    def features(self, ids, att):
        with autocast():
            hs = self.model(input_ids=ids, attention_mask=att, output_hidden_states=True, use_cache=False).hidden_states
        m = att.unsqueeze(-1).float()
        m[:, 0] = 0
        return torch.cat([nm((hs[l].float() * m).sum(1) / m.sum(1)) for l, nm in zip(self.layers, self.norms)], -1)

    def forward(self, ids, att, y, same, cw):
        z = self.features(ids, att)
        zt = z.view(z.shape[0] // 3, 3, -1)
        zc = (zt - zt.mean(1, keepdim=True)).reshape(z.shape[0], -1)
        return wce(self.plain(self.drop(z)), y, cw) + wce(self.centered(self.drop(zc)), y, cw, same.repeat_interleave(3))


def wce(logits, y, cw, mask=None):
    nll = F.cross_entropy(logits.float(), y, reduction='none')
    w = cw[y] if cw is not None else torch.ones_like(nll)
    if mask is not None:
        w = w * mask
    return (nll * w).sum() / w.sum().clamp(min=1e-6)


def encode(tok, texts, max_len):
    """infer.py's tokenisation: default special tokens, truncation at max_len, no end token."""
    return tok(list(texts), truncation=True, max_length=max_len)['input_ids']


def batch(enc, idx, pad):
    Lm = max(len(enc[i]) for i in idx)
    ids = torch.full((len(idx), Lm), pad, dtype=torch.long); att = torch.zeros((len(idx), Lm), dtype=torch.long)
    for r, i in enumerate(idx):
        ids[r, :len(enc[i])] = torch.tensor(enc[i]); att[r, :len(enc[i])] = 1
    return ids.to(DEV), att.to(DEV)


def triplets(groups, rows, rng):
    """Same-group triplets over `rows` (a short last chunk is filled with other members of its group); rows whose
    group has one member go into mixed triplets, flagged 0 so their centred loss is skipped."""
    by = {}
    for r in rows:
        by.setdefault(groups[r], []).append(int(r))
    out, solo = [], []
    for g in sorted(by):
        m = [int(x) for x in rng.permutation(by[g])]
        if len(m) == 1:
            solo.append(m[0]); continue
        for s in range(0, len(m), 3):
            b = m[s:s + 3]
            while len(b) < 3:
                b.append(int(rng.choice([x for x in m if x not in b] or m)))
            out.append((b, 1.0))
    solo = [int(x) for x in rng.permutation(solo)]
    out += [(solo[s:s + 3], 0.0) for s in range(0, len(solo) - 2, 3)]
    return [out[i] for i in rng.permutation(len(out))]


def micro_batches(trips, enc, tri, rng, window=64):
    """`tri` triplets per micro-batch, length-sorted within windows of the shuffled list, then shuffled again."""
    mbs = []
    for s in range(0, len(trips), window):
        w = sorted(trips[s:s + window], key=lambda t: max(len(enc[i]) for i in t[0]))
        mbs += [w[i:i + tri] for i in range(0, len(w) - tri + 1, tri)]
    return [mbs[i] for i in rng.permutation(len(mbs))]


def lr_mult(step, total, warm):
    if step < warm:
        return (step + 1) / warm
    return 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warm) / max(1, total - warm))))


def optimizer(net, a):
    ad = [p for p in net.model.parameters() if p.requires_grad]
    hd = list(net.norms.parameters()) + list(net.plain.parameters()) + list(net.centered.parameters())
    opt = torch.optim.AdamW([{'params': ad, 'lr': a.lr}, {'params': hd, 'lr': a.head_lr}], weight_decay=a.wd)
    return opt, ad + hd


def checkpointed(model):
    """Decoder layers that will run gradient checkpointing (the flag is set and the layer is in training mode)."""
    n = sum(1 for m_ in model.modules() if getattr(m_, 'gradient_checkpointing', False) is True and m_.training)
    assert n > 0, 'gradient checkpointing is off: the decoder would keep every activation'
    return n


def zero_loss(params):
    """A loss of exactly zero that reaches every trainable parameter: stands in for a micro-batch whose fp16 forward
    overflowed, so every rank still runs the same backward (and gradient all-reduce)."""
    return sum(p.sum() for p in params) * 0.0


@torch.no_grad()
def predict(net, enc, rows, pad, bs=16):
    net.eval()
    out = np.zeros((len(rows), net.plain.out_features), np.float32)
    order = np.argsort([len(enc[i]) for i in rows], kind='stable')
    for s in range(0, len(order), bs):
        pos = order[s:s + bs]
        ids, att = batch(enc, [rows[i] for i in pos], pad)
        out[pos] = F.log_softmax(net.plain(net.features(ids, att)), -1).float().cpu().numpy()
    net.train()
    return out


def save(peft_model, net, out, meta):
    os.makedirs(out, exist_ok=True)
    peft_model.save_pretrained(out)
    H = {}
    for i, nm in enumerate(net.norms):
        H[f'norms.{i}.weight'] = nm.weight.detach().float().cpu().clone(); H[f'norms.{i}.bias'] = nm.bias.detach().float().cpu().clone()
    for h in ('plain', 'centered'):
        H[f'{h}.weight'] = getattr(net, h).weight.detach().float().cpu().clone()
        H[f'{h}.bias'] = getattr(net, h).bias.detach().float().cpu().clone()
    torch.save(H, os.path.join(out, 'heads.pt'))
    json.dump(meta, open(os.path.join(out, 'meta.json'), 'w'), indent=1)


def load_heads(net, d):
    H = torch.load(os.path.join(d, 'heads.pt'), map_location='cpu')
    with torch.no_grad():
        for i, nm in enumerate(net.norms):
            nm.weight.copy_(H[f'norms.{i}.weight']); nm.bias.copy_(H[f'norms.{i}.bias'])
        for h in ('plain', 'centered'):
            getattr(net, h).weight.copy_(H[f'{h}.weight']); getattr(net, h).bias.copy_(H[f'{h}.bias'])


def step_fn(net, wrapped, opt, scaler, params, a):
    """One micro-batch: forward (loss), backward; returns (loss value, overflowed)."""
    def run(mb, enc, y_all, pad, cw, sync):
        idx = [i for t, _ in mb for i in t]
        ids, att = batch(enc, idx, pad)
        y = torch.as_tensor(y_all[idx], device=DEV)
        same = torch.tensor([s for _, s in mb], device=DEV, dtype=torch.float32)
        ctx = wrapped.no_sync() if (WORLD > 1 and not sync) else contextlib.nullcontext()
        with ctx:
            loss = wrapped(ids, att, y, same, cw)
            bad = not bool(torch.isfinite(loss))
            if bad:
                loss = zero_loss(params)
            scaler.scale(loss / a.accum).backward()
        return float(loss.detach()), bad
    return run


def stage0(a):
    """fp16 health (non-finite pooled depths on the longest pool texts), peak memory of a full-length training step,
    and training speed on random triplets. Writes <out>/check.json."""
    pool = pd.read_parquet(a.pool)
    tok, m, hid = load_base(a)
    model, n_lora = add_lora(m, a)
    net = Net(model, hid, a.layers, len(a.labels)).to(DEV)
    enc = encode(tok, pool.text.astype(str).tolist(), a.max_len)
    lens = np.array([len(x) for x in enc])
    longest = [int(i) for i in np.argsort(-lens)[:a.check_rows]]
    bad_rows = 0
    with torch.no_grad():
        net.eval()
        for s in range(0, len(longest), 6):
            ids, att = batch(enc, longest[s:s + 6], tok.pad_token_id)
            with autocast():
                hs = model(input_ids=ids, attention_mask=att, output_hidden_states=True, use_cache=False).hidden_states
            fin = torch.stack([torch.isfinite(hs[l]).flatten(1).all(1) for l in a.layers]).all(0)
            bad_rows += int((~fin).sum())
        net.train()
    log(f'fp16 check: {bad_rows}/{len(longest)} of the longest pool texts overflow at a pooled depth; '
        f'checkpointed layers {checkpointed(model)}')
    y_all = pool.model.map({l: i for i, l in enumerate(a.labels)}).fillna(0).astype(int).values
    opt, params = optimizer(net, a)
    scaler = torch.amp.GradScaler('cuda', init_scale=256.0, enabled=CUDA)
    run = step_fn(net, net, opt, scaler, params, a)
    if CUDA:
        torch.cuda.reset_peak_memory_stats(DEV)
    t = time.time()
    trips = [([longest[3 * k], longest[3 * k + 1], longest[3 * k + 2]], 1.0) for k in range(4)]
    tri = a.tri
    while True:   # full-length micro-batches: the memory high-water mark; halve the micro-batch on running out
        try:
            for k in range(2):
                run(trips[2 * k:2 * k + tri], enc, y_all, tok.pad_token_id, None, True)
            break
        except torch.cuda.OutOfMemoryError:
            opt.zero_grad(set_to_none=True); torch.cuda.empty_cache()
            assert tri > 1, 'out of GPU memory at one triplet per micro-batch'
            log(f'out of memory at {tri} triplets per micro-batch; trying {tri // 2}'); tri //= 2
    a.tri = tri
    peak = torch.cuda.max_memory_allocated(DEV) / 2 ** 30 if CUDA else 0.0
    opt.zero_grad(set_to_none=True)
    rng = np.random.RandomState(a.seed)
    groups = pool.prompt_key.astype(str).values
    mbs = micro_batches(triplets(groups, rng.choice(len(pool), min(3000, len(pool)), replace=False), rng), enc, a.tri, rng)[:12]
    if CUDA:
        torch.cuda.synchronize()
    t = time.time(); toks = 0; bad = 0
    for mb in mbs:
        _, b = run(mb, enc, y_all, tok.pad_token_id, None, True); bad += b
        toks += sum(len(enc[i]) for t_, _ in mb for i in t_)
    if CUDA:
        torch.cuda.synchronize()
    tps = toks / (time.time() - t)
    total_tokens = int(lens.sum())
    res = dict(ok=bad_rows <= 0.02 * len(longest) and bad == 0, overflow_rows=bad_rows, rows_checked=len(longest),
               peak_gib=round(peak, 2), tokens_per_s_one_gpu=round(tps, 1), lora_params=n_lora, pool_tokens=total_tokens,
               hours_per_epoch_two_gpus=round(total_tokens / (2 * tps) / 3600, 2), train_step_overflow=bad, hidden=hid,
               tri=tri)
    log('check', json.dumps(res))
    os.makedirs(a.out, exist_ok=True)
    json.dump(res, open(os.path.join(a.out, 'check.json'), 'w'), indent=1)


def stage1(a):
    if CUDA:
        torch.cuda.set_device(DEV)
    torch.manual_seed(a.seed); np.random.seed(a.seed)
    pool = pd.read_parquet(a.pool)
    y_all = pool.model.map({l: i for i, l in enumerate(a.labels)}).values
    assert not np.isnan(y_all.astype(float)).any(), 'pool label outside --labels'
    y_all = y_all.astype(int)
    tok, m, hid = load_base(a)
    model, n_lora = add_lora(m, a, a.init)   # before the process group: once one exists, peft loads an adapter on its tensor-parallel path
    if WORLD > 1:
        torch.distributed.init_process_group('nccl' if CUDA else 'gloo')
    net = Net(model, hid, a.layers, len(a.labels)).to(DEV)
    prior = 0.0
    if a.init:   # continue an earlier stage-1 run: its adapter and heads, a new triplet order (--seed), a new cosine cycle
        load_heads(net, a.init)
        prev = json.load(open(os.path.join(a.init, 'meta.json')))
        prior = prev.get('epochs_total', prev['epochs_done'])
    net.train()   # from_pretrained returns eval mode, in which the decoder layers skip gradient checkpointing
    n_ckpt = checkpointed(model)
    wrapped = net
    if WORLD > 1:
        wrapped = nn.parallel.DistributedDataParallel(net, device_ids=[LOCAL] if CUDA else None,
                                                      gradient_as_bucket_view=True)
    enc = encode(tok, pool.text.astype(str).tolist(), a.max_len)
    counts = np.bincount(y_all, minlength=len(a.labels)).astype(float)
    cw = torch.tensor(len(y_all) / (len(a.labels) * np.maximum(counts, 1)), dtype=torch.float32, device=DEV) \
        if a.class_weights else None
    groups = pool.prompt_key.astype(str).values
    rng = np.random.RandomState(a.seed)
    mbs = []
    for e in range(int(math.ceil(a.epochs))):
        ep = micro_batches(triplets(groups, np.arange(len(pool)), rng), enc, a.tri, rng)
        if e + 1 > a.epochs:
            ep = ep[:int(len(ep) * (a.epochs - e))]
        mbs += ep
    mine = mbs[RANK::WORLD][:len(mbs) // WORLD]
    planned = len(mine) // a.accum
    log(f'stage 1: {len(pool)} pool rows, {len(mbs)} micro-batches of {a.tri} triplets over {a.epochs} epochs, '
        f'{planned} optimizer steps per rank ({WORLD} ranks x {a.accum} accumulation), LoRA params {n_lora}, '
        f'class weights {"on" if cw is not None else "off"}, checkpointed layers {n_ckpt}')
    opt, params = optimizer(net, a)
    scaler = torch.amp.GradScaler('cuda', init_scale=256.0, enabled=CUDA)
    run = step_fn(net, wrapped, opt, scaler, params, a)
    total, warm = planned, max(1, int(a.warmup * planned))
    budget = a.hours * 3600; t_train = time.time(); last_ckpt = time.time()
    step, losses, bad, skipped, t_cal = 0, [], 0, 0, None
    meta = dict(stage=1, base=a.base, keep=a.keep, layers=a.layers, max_len=a.max_len, labels=a.labels, r=a.r,
                alpha=a.alpha, lr=a.lr, head_lr=a.head_lr, epochs=a.epochs, tri=a.tri, accum=a.accum, world=WORLD,
                class_weights=cw is not None, pool_rows=len(pool), init=a.init, seed=a.seed)
    for i, mb in enumerate(mine):
        sync = (i + 1) % a.accum == 0
        lv, b = run(mb, enc, y_all, tok.pad_token_id, cw, sync)
        losses.append(lv); bad += b
        if not sync:
            continue
        for g, base_lr in zip(opt.param_groups, (a.lr, a.head_lr)):
            g['lr'] = base_lr * lr_mult(step, total, warm)
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        before = scaler.get_scale(); scaler.step(opt); scaler.update()
        skipped += scaler.get_scale() < before
        opt.zero_grad(set_to_none=True); step += 1
        if step == 10:
            t_cal = time.time()
        if step == 60 or (step > 60 and step % 200 == 0):
            # the time box: steps that fit the remaining budget at the measured speed (rank 0 decides for all)
            sps = (time.time() - t_cal) / (step - 10)
            fit = step + int((budget - (time.time() - t_train)) / sps)
            new_total = torch.tensor([min(planned, max(step + 1, fit))], device=DEV)
            if WORLD > 1:
                torch.distributed.broadcast(new_total, 0)
            if step == 60:
                total = int(new_total.item()); warm = min(warm, max(1, total // 20))
                log(f'speed {sps:.2f} s/step: {total} of {planned} planned steps fit the {a.hours} h budget '
                    f'({total / planned * a.epochs:.2f} epochs)')
            elif int(new_total.item()) < total:
                total = int(new_total.item()); log(f'running slower: total cut to {total} steps')
        if step % 100 == 0:
            log(f'step {step}/{total} loss {np.mean(losses[-100 * a.accum:]):.4f} overflow {bad} skipped {skipped} '
                f'lr {opt.param_groups[0]["lr"]:.2e} ({(time.time() - t_train) / 3600:.2f} h)')
        if RANK == 0 and time.time() - last_ckpt > a.ckpt_min * 60:
            save(model, net, os.path.join(a.out, 'ckpt'), dict(meta, step=step, total=total)); last_ckpt = time.time()
        if step >= total:
            break
    hours = (time.time() - t_train) / 3600
    meta.update(steps=step, total_steps=total, planned_steps=planned, hours=round(hours, 3), overflow_microbatches=bad,
                skipped_steps=int(skipped), final_loss=float(np.mean(losses[-100 * a.accum:])),
                epochs_done=round(step / planned * a.epochs, 3))
    meta['epochs_total'] = round(prior + meta['epochs_done'], 3)   # pool epochs including the continued run's
    if RANK == 0:
        save(model, net, a.out, meta)
        log(f'stage 1 done: {step} steps in {hours:.2f} h, {meta["epochs_done"]} epochs, overflow micro-batches {bad}, '
            f'skipped steps {skipped}; saved {a.out}')
    if WORLD > 1:
        torch.distributed.barrier(); torch.distributed.destroy_process_group()


def stage2(a):
    torch.manual_seed(a.seed)
    os.makedirs(os.path.dirname(a.out.rstrip('/')) or '.', exist_ok=True)
    tr = pd.read_parquet(a.train)
    y_all = tr.label.astype(int).values; folds = tr.fold.astype(int).values
    tok, m, hid = load_base(a)
    model, n_lora = add_lora(m, a, a.init)
    net = Net(model, hid, a.layers, len(a.labels)).to(DEV)
    from peft import get_peft_model_state_dict, set_peft_model_state_dict
    s1 = {k: v.detach().clone() for k, v in get_peft_model_state_dict(model).items()}
    enc = encode(tok, tr.text.astype(str).tolist(), a.max_len)
    groups = tr.prompt_id.astype(str).values
    s1_meta = json.load(open(os.path.join(a.init, 'meta.json')))
    if '0' in a.jobs.split(','):   # the pool-only model on the official rows, none of which it has seen
        lp = predict(net, enc, list(range(len(tr))), tok.pad_token_id)
        s1_meta['official_balanced_accuracy'] = float(np.mean([(lp[y_all == c].argmax(1) == c).mean() for c in np.unique(y_all)]))
        np.save(a.out + '_stage1_on_train.npy', lp)
        log(f"stage-1 model on the official rows: balanced accuracy {s1_meta['official_balanced_accuracy']:.4f}")
    for job in a.jobs.split(','):
        t0 = time.time()
        set_peft_model_state_dict(model, s1); load_heads(net, a.init)
        rows = np.arange(len(tr)) if job == 'full' else np.where(folds != int(job))[0]
        rng = np.random.RandomState(a.seed + (99 if job == 'full' else int(job)))
        mbs = []
        for e in range(int(math.ceil(a.epochs))):
            mbs += micro_batches(triplets(groups, rows, rng), enc, a.tri, rng)
        total = len(mbs) // a.accum; warm = max(1, int(a.warmup * total))
        opt, params = optimizer(net, a)
        scaler = torch.amp.GradScaler('cuda', init_scale=256.0, enabled=CUDA)
        run = step_fn(net, net, opt, scaler, params, a)
        net.train(); checkpointed(model)   # see stage1: gradient checkpointing only runs in training mode
        step, losses, bad, skipped = 0, [], 0, 0
        for i, mb in enumerate(mbs[:total * a.accum]):
            lv, b = run(mb, enc, y_all, tok.pad_token_id, None, True)
            losses.append(lv); bad += b
            if (i + 1) % a.accum:
                continue
            for g, base_lr in zip(opt.param_groups, (a.lr, a.head_lr)):
                g['lr'] = base_lr * lr_mult(step, total, warm)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            before = scaler.get_scale(); scaler.step(opt); scaler.update()
            skipped += scaler.get_scale() < before
            opt.zero_grad(set_to_none=True); step += 1
            if step % 50 == 0:
                log(f'job {job} step {step}/{total} loss {np.mean(losses[-50 * a.accum:]):.4f} overflow {bad} skipped {skipped}')
        name = os.path.basename(a.out.rstrip('/')) + ('' if job == 'full' else f'_f{job}')
        out = os.path.join(os.path.dirname(a.out.rstrip('/')), name)
        check = predict(net, enc, list(range(40)), tok.pad_token_id)
        meta = dict(s1_meta, stage=2, job=job, steps=step, epochs=a.epochs, lr=a.lr, head_lr=a.head_lr, tri=a.tri,
                    accum=a.accum, overflow_microbatches=bad, skipped_steps=int(skipped), init=a.init,
                    final_loss=float(np.mean(losses[-50 * a.accum:])), minutes=round((time.time() - t0) / 60, 2),
                    in_sample_check_40=float((check.argmax(1) == y_all[:40]).mean()))
        if job != 'full':
            valid = np.where(folds == int(job))[0]
            lp = predict(net, enc, list(valid), tok.pad_token_id)
            meta['valid_accuracy'] = float((lp.argmax(1) == y_all[valid]).mean())
            meta['valid_balanced_accuracy'] = float(np.mean([(lp[y_all[valid] == c].argmax(1) == c).mean()
                                                            for c in np.unique(y_all[valid])]))
        save(model, net, out, meta)
        np.save(os.path.join(out, 'check40.npy'), check)
        if job != 'full':
            np.savez(os.path.join(out, 'valid.npz'), ids=np.array(tr.id.astype(str).values[valid].tolist()), logits=lp)
        log(f"job {job} done: {step} steps, {meta['minutes']} min, in-sample 40 {meta['in_sample_check_40']:.2f}"
            + (f", held-out accuracy {meta['valid_accuracy']:.4f} balanced {meta['valid_balanced_accuracy']:.4f}"
               if job != 'full' else '') + f'; saved {out}')


if __name__ == '__main__':
    args = parse()
    {0: stage0, 1: stage1, 2: stage2}[args.stage](args)

In [ ]:
%%writefile /kaggle/working/nlm_oof.py
"""Out-of-fold log-probabilities of fold-averaged LoRA24 members (nlm-v27's q8v5, q14v5, ll8v5; Dhruv's q25_14v5) on train.csv.

    python nlm_oof.py --specs '<json list>' --out oof.npz [--hypothesis folds.npy] [--folds folds.npy] [--enc_dir DIR]

Each member ships five fold adapters (stage 2 without one fold of the 2,699 official rows) and a full fit, so a fold
adapter's predictions on its held-out fold are out of fold. The split is not published. Without --folds, the first
member (q8v5, the cheapest) scores every train row with every fold adapter, and each prompt goes to the adapter that
gives its true labels the lowest summed log-probability: the one that never saw it. With --folds (the split inferred
by an earlier run) that step is skipped. Every member then scores only the rows each adapter held out. The plain head
only (per passage, no prompt-group twin or Gaussian), as most hidden-test passages are singletons. Feature extraction
is infer.py's l24_predict: truncated base in float16, LoRA merged into the weights, masked mean pool (first token
dropped) of 8 depths, one LayerNorm each, bfloat16 redo of overflowing batches.

A member whose 8 depths are not published (spec 'search': true) gets them from its full-data adapter (spec 'full'):
every hidden state of the first 256 train rows, which that adapter was trained on, is pooled once, and the depth set
whose plain head gives those rows the highest mean true-label log-probability wins (named candidates, then coordinate
ascent one slot at a time). --enc_dir also writes each member's OOF as <name>/oof.npz (ids, labels, logits), the
format experiments/exp015_nlm_core_oof.py reads for candidates.
"""
import argparse
import contextlib
import gc
import json
import os
import time

import numpy as np
import pandas as pd
import torch

LABELS = ['claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash',
          'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max']
T0 = time.time()
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False


def log(*x):
    print(f'[{time.time() - T0:7.0f}s]', *x, flush=True)


def lsm(z):
    return z - z.max(1, keepdims=True) - np.log(np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True))


def load_base(spec):
    from transformers import AutoConfig, AutoModel, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(spec['path'])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    cfg = AutoConfig.from_pretrained(spec['path']); tc = cfg.text_config if hasattr(cfg, 'text_config') else cfg
    tc.num_hidden_layers = spec['keep']; tc.use_cache = False
    if getattr(tc, 'layer_types', None) is not None:
        tc.layer_types = tc.layer_types[:spec['keep']]
    m = AutoModel.from_pretrained(spec['path'], config=cfg, dtype=torch.float16, device_map='auto').eval()
    return tok, m, dict(m.named_modules()), next(m.parameters()).device


@contextlib.contextmanager
def merged(mods, ad):
    """infer.py's in-place LoRA merge (W += B A * alpha / r) for the duration of the block; yields the adapter's heads."""
    from safetensors.torch import load_file
    adp = os.path.join(ad, 'adapter') if os.path.exists(os.path.join(ad, 'adapter', 'adapter_config.json')) else ad
    acfg = json.load(open(os.path.join(adp, 'adapter_config.json'))); scale = acfg['lora_alpha'] / acfg['r']
    sd = load_file(os.path.join(adp, 'adapter_model.safetensors')); deltas = []
    with torch.no_grad():
        for k, v in sd.items():
            if '.lora_A.' not in k:
                continue
            name = k.replace('base_model.model.', '', 1).split('.lora_A.')[0]
            if name not in mods and ('model.' + name) in mods:
                name = 'model.' + name
            W = mods[name].weight; B = sd[k.replace('.lora_A.', '.lora_B.')]
            W += ((B.to(W.device, torch.float32) @ v.to(W.device, torch.float32)) * scale).to(W.dtype)
            deltas.append((W, B, v))
    H = torch.load(os.path.join(ad, 'heads.pt'), map_location='cpu')
    if os.path.exists(os.path.join(ad, 'meta.json')):
        perm = [json.load(open(os.path.join(ad, 'meta.json')))['labels'].index(l) for l in LABELS]
        H['plain.weight'], H['plain.bias'] = H['plain.weight'][perm], H['plain.bias'][perm]
    try:
        yield H
    finally:
        with torch.no_grad():
            for W, B, A_ in deltas:
                W -= ((B.to(W.device, torch.float32) @ A_.to(W.device, torch.float32)) * scale).to(W.dtype)
        del deltas, sd; gc.collect(); torch.cuda.empty_cache()


@torch.no_grad()
def pooled(spec, tok, m, first, texts, depths, ovf):
    """Masked mean pool (first token dropped) of the hidden states at `depths`, float32 on the CPU: {depth: (n, hid)}."""
    enc = tok(list(texts), truncation=True, max_length=spec.get('max_len', 384))['input_ids']
    order = np.argsort([len(x) for x in enc]); bs = spec.get('bs', 16); out = {}
    for i in range(0, len(order), bs):
        b = order[i:i + bs]; Lm = max(len(enc[j]) for j in b)
        ids = torch.full((len(b), Lm), tok.pad_token_id); att = torch.zeros((len(b), Lm), dtype=torch.long)
        for r, j in enumerate(b):
            ids[r, :len(enc[j])] = torch.tensor(enc[j]); att[r, :len(enc[j])] = 1
        hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states
        if not all(torch.isfinite(hs[l]).all() for l in depths):
            m.to(torch.bfloat16); hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states; m.to(torch.float16); ovf[0] += 1
        mask = att.unsqueeze(-1).float(); mask[:, 0] = 0
        for l in depths:
            out.setdefault(l, np.zeros((len(enc), hs[l].shape[-1]), np.float32))[b] = ((hs[l].float().cpu() * mask).sum(1) / mask.sum(1)).numpy()
    return out


def head_lp(H, feats, L):
    """infer.py's plain head: one LayerNorm per depth, concatenated, linear, log-softmax."""
    hid = H['norms.0.weight'].shape[0]
    with torch.no_grad():
        Z = []
        for i, l in enumerate(L):
            nm = torch.nn.LayerNorm(hid); nm.load_state_dict({'weight': H[f'norms.{i}.weight'], 'bias': H[f'norms.{i}.bias']})
            Z.append(nm(torch.from_numpy(feats[l])).numpy())
    return lsm(np.concatenate(Z, -1) @ H['plain.weight'].float().numpy().T + H['plain.bias'].float().numpy())


def search_layers(spec, tok, m, mods, first, texts, y, rows=256):
    """The 8 depths of a member whose depths are not published, from its full-data adapter's fit to rows it saw."""
    ovf = [0]
    with merged(mods, spec['full']) as H:
        depths = list(range(1, spec['keep'] + 1))
        F = pooled(spec, tok, m, first, texts[:rows], depths, ovf)
    yy = y[:rows]

    def fit(L):
        lp = head_lp(H, F, L)
        return float(lp[np.arange(len(yy)), yy].mean()), float((lp.argmax(1) == yy).mean())
    k = spec['keep']
    cands = {'nlm-v27 L24': [2, 4, 6, 8, 12, 16, 20, 24], 'every 4th': [4, 8, 12, 16, 20, 24, 28, 32],
             'L24 + last': [2, 4, 6, 8, 12, 16, 20, k], 'scaled to keep': sorted({max(1, round(x * k / 24)) for x in [2, 4, 6, 8, 12, 16, 20, 24]})}
    cands = {n: L for n, L in cands.items() if len(L) == 8 and max(L) <= k}
    scored = {n: fit(L) for n, L in cands.items()}
    for n, (ll, acc) in scored.items():
        log(f"  depth search {spec['name']}: {n} {cands[n]}: mean true-label log-prob {ll:.3f}, accuracy {acc:.3f}")
    best = list(cands[max(scored, key=lambda n: scored[n][0])]); cur = fit(best)[0]
    for _ in range(2):   # coordinate ascent, one slot at a time
        for i in range(8):
            for d in depths:
                if d in best:
                    continue
                L = best[:i] + [d] + best[i + 1:]
                s = fit(L)[0]
                if s > cur + 1e-6:
                    best, cur = L, s
    ll, acc = fit(best)
    log(f"  depth search {spec['name']}: chosen {best}: mean true-label log-prob {ll:.3f}, accuracy {acc:.3f} "
        f"(rows {len(yy)}, bf16 redo batches {ovf[0]})")
    return best, {'candidates': {n: [cands[n], *scored[n]] for n in cands}, 'chosen': [best, ll, acc]}


def member(spec, texts_per_adapter, loaded=None):
    """Plain-head log-probabilities of each fold adapter on its own list of texts (infer.py's l24_predict)."""
    tok, m, mods, first = loaded or load_base(spec)
    outs, ovf = [], [0]
    for ad, texts in zip(spec['adapters'], texts_per_adapter):
        if not len(texts):
            outs.append(np.zeros((0, 18), np.float32)); continue
        with merged(mods, ad) as H:
            F = pooled(spec, tok, m, first, texts, spec['layers'], ovf)
            outs.append(head_lp(H, F, spec['layers']))
        log(f"  {spec['name']} {os.path.basename(ad)}: {len(texts)} rows (bf16 redo batches {ovf[0]})")
    del m, mods; gc.collect(); torch.cuda.empty_cache()
    return outs


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--data', required=True); p.add_argument('--specs', required=True); p.add_argument('--out', required=True)
    p.add_argument('--hypothesis', default=''); p.add_argument('--folds', default=''); p.add_argument('--enc_dir', default='')
    a = p.parse_args()
    train = pd.read_csv(os.path.join(a.data, 'train.csv'))
    train['text'] = train.text.fillna('')
    y = train.model.map({l: i for i, l in enumerate(LABELS)}).values
    texts = train.text.values
    specs = json.loads(a.specs)
    out = dict(ids=train.id.astype(str).values, labels=np.array(LABELS), y=y)
    if a.folds:
        folds = np.load(a.folds); rest = specs
        assert len(folds) == len(train) and pd.Series(folds).groupby(train.prompt_id.values).nunique().max() == 1
        log(f'folds given: sizes {np.bincount(folds, minlength=5).tolist()}')
    else:
        # 1. the first member scores every row with every fold adapter; prompts go to the adapter that never saw them
        first = specs[0]
        allk = np.stack(member(first, [texts] * 5))                       # (5, N, 18)
        truth = allk[:, np.arange(len(y)), y]                              # (5, N) true-label log-probability
        prompts, pinv = np.unique(train.prompt_id.values, return_inverse=True)
        per_prompt = np.stack([np.bincount(pinv, weights=truth[k], minlength=len(prompts)) for k in range(5)])
        pfold = per_prompt.argmin(0); folds = pfold[pinv]
        srt = np.sort(per_prompt, 0); gap = (srt[1] - srt[0]) / np.bincount(pinv)   # per-row nats between the two lowest
        log(f'inferred folds: sizes {np.bincount(folds, minlength=5).tolist()}, per-row gap min {gap.min():.2f} '
            f'median {np.median(gap):.2f} nats')
        acc_in = [(allk[k][folds != k].argmax(1) == y[folds != k]).mean() for k in range(5)]
        acc_out = [(allk[k][folds == k].argmax(1) == y[folds == k]).mean() for k in range(5)]
        log('fold adapter accuracy, rows it trained on vs held out: ' +
            ', '.join(f'f{k} {i:.3f}/{o:.3f}' for k, (i, o) in enumerate(zip(acc_in, acc_out))))
        if a.hypothesis:
            h = np.load(a.hypothesis)
            # fold labels are arbitrary: compare the partitions
            same = np.mean([[np.array_equal(folds == i, h == j) for j in range(5)] for i in range(5)], dtype=float) * 5
            log(f'GroupKFold hypothesis: {same:.0%} of inferred folds match a hypothesis fold exactly')
        out['gap'] = gap
        out[f"oof_{first['name']}"] = allk[folds, np.arange(len(y))]
        out[f"all5_{first['name']}"] = allk.astype(np.float16)
        rest = specs[1:]
    out['folds'] = folds
    # 2. the other members: each fold adapter on the rows it held out
    search = {}
    for spec in rest:
        loaded = load_base(spec)
        if spec.get('search'):
            spec['layers'], search[spec['name']] = search_layers(spec, *loaded, texts, y)
        lp = member(spec, [texts[folds == k] for k in range(5)], loaded)
        del loaded; gc.collect(); torch.cuda.empty_cache()
        oof = np.zeros((len(y), 18), np.float32)
        for k in range(5):
            oof[folds == k] = lp[k]
            log(f"  {spec['name']} fold {k}: held-out accuracy {(lp[k].argmax(1) == y[folds == k]).mean():.3f}")
        out[f"oof_{spec['name']}"] = oof
    for k_, v in out.items():
        if k_.startswith('oof_'):
            rec = [(v[y == c].argmax(1) == c).mean() for c in range(18)]
            log(f'{k_}: OOF balanced accuracy {np.mean(rec):.4f}')
            if a.enc_dir:
                d = os.path.join(a.enc_dir, k_[4:]); os.makedirs(d, exist_ok=True)
                np.savez(os.path.join(d, 'oof.npz'), ids=np.array(train.id.astype(str).tolist()), labels=np.array(LABELS),
                         logits=v.astype(np.float32))
    if search:
        json.dump(search, open(os.path.splitext(a.out)[0] + '_depths.json', 'w'), indent=1)
    np.savez(a.out, **out)
    log('NLM OOF DONE', a.out)


if __name__ == '__main__':
    main()

In [ ]:
# ---- driver cell (inserted by kaggle/member/build_member.py, which fills CONFIG)
# Two-stage LoRA24 member (l24_worker.py), in the layout of Dhruv Tibarewal's nlm-l24-adapters-v5.
# Stage 1 (CONFIG['stage'] == 1): an fp16 health / memory / speed check of each candidate base in turn (the first that
# passes is trained), then the pool stage on Dhruv's 120k public pool (nlm_pool_v5.parquet, which already excludes
# every train text and near-duplicate), data-parallel over both T4s and time-boxed. Output: /kaggle/working/<tag>/.
# Stage 2 (CONFIG['stage'] == 2): from that output, the official rows once per held-out fold (the split inferred from
# nlm-v27's own fold adapters, so out-of-fold predictions line up with exp015's replica) and once on all rows, three
# jobs per GPU; then the out-of-fold log-probs (enc/<tag>/oof.npz) and a reload check of the full-data adapter through
# infer.py's own merge path (nlm_oof.py).
import glob, hashlib, json, os, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd

CONFIG = {'kernel': 'ntl-l24-glm9-s2', 'stage': 2, 'labels': ['claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max'], 'train_sha256': '40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707', 'tag': 'glm9v1', 'folds': '02422223301021244022120134014321033301232224132403011222023013124112112133304440121301303144412002003034241010444304241021113043100440434012002410414401212220402420302403410131040433323303302432102304012203132321340412223100411400342432443420114213333132040431304343342014311404242214340201101300444040222420202041201211032011002224201241321002244411104442320301040204032124323222340312400223011223043340040402401310013230130220303414222443404042212423002003413144211341210124323144424321111100331402130031430014210243332310201013220010233443242102324231331421304402244444032400310414434422214413221414214221302114302442120430312223330413240434343002122242111244113034303222300124011414301023420322043411404310213202011344404220402441444230330230214144434403222213320423022021243001001224343140401113230113042140310213303401411422042111214402310423144403022331134113443303343022131314033314012233401320113033040143343314002030213324214230101030424422204213442300321021403203013403414300422224021123214102440202302412410433030241410210130220110432101420201331331224242143002220133042223133013211011313024004413020112333211240122320422130433211214340242213303104120111042313101133021343124201022321103434210324102132302004111220434134042100240140413232142012120434404141441234113202342401244113103402314134044001243410411041123141324433301232202404340032322144202403110113014103012434301301331230342012232341401242040020144243022223112324223120041343311404202343123014314304044230430413113204004411231022413243230232423300412140122022144122202034001412123411042420434004400010304423401011331043214333214433122431130142234022314424301421241042421433413431134233444432033123442310334104223401313213413334003040010211432123223102311031332143011244310212104334031021404032032442312300224021121223431241144304142122020414442411110321303144021430331140114341440432101120330413321310214132132420134212444204210313421004112104112410221041243134331300422331200031403112342142003311313121324214320114213203002043440111033134142223410413413143000244020434432144314402044143313024342401021430334144132040001134112130330213310313303101323242230202100342433333230322304123420243114443310340233004212401211301330031124342323031304324440111141424312000140013024131433043420230213100044430314210331122231121020400204003424202220002110040403220132440012442303034323132044042132034410003202311000413033323311131432040333242131041143142300422121040203003341334034100132001340233432443214040234130243142303240011300230242021114310031041322024202300000232114104202422123112202134014231301101400334030343102310304323043043334011304122014220122230230121433303103040301244203013103042041414032420430132241441241032102144344422', 'gpu_jobs': ['0,2,4', '1,3,full'], 'args': ['--keep', '24', '--layers', '2,4,6,8,12,16,20,24', '--max-len', '384', '--tri', '2', '--accum', '3', '--lr', '5e-5', '--head-lr', '5e-4', '--warmup', '0.1', '--epochs', '2']}
WORK, TMP = Path("/kaggle/working"), Path("/tmp/l24")
STARTED = time.time()


def sh(command):
    return subprocess.run(command, shell=True, capture_output=True, text=True).stdout.strip()


def hours():
    return f"{(time.time() - STARTED) / 3600:.2f} h"


print(sh("nvidia-smi -L"), "\n", sh("df -h /tmp /kaggle/working | tail -2"), flush=True)
import torch, transformers
try:
    import peft
except ImportError:
    print(sh(f"{sys.executable} -m pip install -q peft 2>&1 | tail -2"), flush=True)
    import peft
print(dict(torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__), flush=True)
assert torch.cuda.device_count() == 2, "expected T4 x2"
cfgs = sorted(glob.glob("/kaggle/input/**/config.json", recursive=True))


def mdir(parts):
    hits = [os.path.dirname(c) for c in cfgs if all(p in c.lower() for p in parts)]
    assert hits, parts
    return hits[0]


train_csv = sorted(glob.glob("/kaggle/input/**/name-that-llm/train.csv", recursive=True))[0]
assert hashlib.sha256(Path(train_csv).read_bytes()).hexdigest() == CONFIG["train_sha256"], "train.csv changed"
train = pd.read_csv(train_csv, dtype=str, keep_default_na=False)
assert len(train) == 2699
TMP.mkdir(parents=True, exist_ok=True)
LOGS = WORK / "logs"
LOGS.mkdir(exist_ok=True)
labels = json.dumps(CONFIG["labels"])
worker = [sys.executable, "-u", str(WORK / "l24_worker.py")]
env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false")


def follow(procs, every=600):
    """Wait for the processes, printing the last line of each log every `every` seconds."""
    while any(p.poll() is None for _, p in procs):
        time.sleep(every)
        for name, _ in procs:
            lines = (LOGS / f"{name}.log").read_text(errors="replace").splitlines()
            print(f"[{hours()}] {name}: {lines[-1] if lines else ''}", flush=True)
    for name, p in procs:
        tail = (LOGS / f"{name}.log").read_text(errors="replace").splitlines()
        print(f"==== {name} exit {p.returncode}\n" + "\n".join(tail[-25:]), flush=True)
    return [p.returncode for _, p in procs]


if CONFIG["stage"] == 0:
    # probe: the check of every base (two at a time, one per GPU), then a few minutes of two-GPU stage 1 on a slice
    pool_path = sorted(glob.glob("/kaggle/input/**/nlm_pool_v5.parquet", recursive=True))[0]
    pool = pd.read_parquet(pool_path)
    pool = pool[pool.model.isin(CONFIG["labels"]) & ~pool.text.astype(str).isin(set(train.text))].copy()
    pool[["text", "model", "prompt_key"]].to_parquet(TMP / "pool.parquet", index=False)
    keys = pool.prompt_key.drop_duplicates().sample(400, random_state=0)
    pool[pool.prompt_key.isin(keys)][["text", "model", "prompt_key"]].to_parquet(TMP / "slice.parquet", index=False)
    results = {}
    for s in range(0, len(CONFIG["bases"]), 2):
        procs = []
        for gpu, b in enumerate(CONFIG["bases"][s:s + 2]):
            name = f"check_{b['tag']}"
            args = ["--base", mdir(b["parts"]), "--labels", labels, "--pool", str(TMP / "pool.parquet"), *b["args"]]
            procs.append((name, subprocess.Popen(worker + ["--stage", "0", "--out", str(WORK / name), *args],
                                                 stdout=open(LOGS / f"{name}.log", "w"), stderr=subprocess.STDOUT,
                                                 env=dict(env, CUDA_VISIBLE_DEVICES=str(gpu)))))
        follow(procs, every=120)
        for name, p in procs:
            f = WORK / name / "check.json"
            results[name] = dict(rc=p.returncode, **(json.loads(f.read_text()) if f.exists() else {}))
    print("CHECKS", json.dumps(results, indent=1), flush=True)
    b = next(b for b in CONFIG["bases"] if results[f"check_{b['tag']}"].get("ok"))
    log = open(LOGS / "ddp.log", "w")
    p = subprocess.Popen([sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", "2",
                          str(WORK / "l24_worker.py"), "--stage", "1", "--out", str(WORK / f"ddp_{b['tag']}"),
                          "--hours", "0.05", "--class-weights", "--ckpt-min", "1", "--base", mdir(b["parts"]),
                          "--labels", labels, "--pool", str(TMP / "slice.parquet"), *b["args"]],
                         stdout=log, stderr=subprocess.STDOUT, env=env)
    rc = follow([("ddp", p)], every=60)
    print("PROBE DONE", b["tag"], rc, hours(), flush=True)

if CONFIG["stage"] == 1:
    pool_path = sorted(glob.glob("/kaggle/input/**/nlm_pool_v5.parquet", recursive=True))[0]
    pool = pd.read_parquet(pool_path)
    pool = pool[pool.model.isin(CONFIG["labels"])].copy()
    pool["text"] = pool.text.astype(str)
    overlap = pool.text.isin(set(train.text))
    print(f"pool {pool_path}: {len(pool)} rows, {int(overlap.sum())} exact train texts dropped", flush=True)
    pool[~overlap][["text", "model", "prompt_key"]].to_parquet(TMP / "pool.parquet", index=False)
    chosen = None
    if CONFIG.get("init_tag"):   # continue an earlier stage-1 run of this base (its kernel is attached), no new check
        hits = [os.path.dirname(f) for f in glob.glob(f"/kaggle/input/**/{CONFIG['init_tag']}/meta.json", recursive=True)
                if json.load(open(f)).get("stage") == 1]
        assert len(hits) == 1, hits
        prev = json.load(open(os.path.join(hits[0], "meta.json")))
        done_before = prev.get("epochs_total", prev["epochs_done"])
        b = next(b for b in CONFIG["bases"] if b["tag"] == CONFIG["init_tag"])
        assert os.path.exists(prev["base"]), prev["base"]
        args = ["--base", prev["base"], "--labels", labels, "--pool", str(TMP / "pool.parquet"), *b["args"]]
        for k in ("tri", "accum"):
            args[args.index(f"--{k}") + 1] = str(prev[k])
        args[args.index("--epochs") + 1] = str(round(CONFIG["epochs_total"] - done_before, 3))
        args += ["--init", hits[0], "--seed", "1"]
        print(f"continuing {b['tag']} from {hits[0]} ({done_before} pool epochs done)", flush=True)
        chosen = (b, prev["base"], args)
    for b in [] if chosen else CONFIG["bases"]:
        path = mdir(b["parts"])
        args = ["--base", path, "--labels", labels, "--pool", str(TMP / "pool.parquet"), *b["args"]]
        name = f"check_{b['tag']}"
        with open(LOGS / f"{name}.log", "w") as log:
            rc = subprocess.run(worker + ["--stage", "0", "--out", str(WORK / name), *args], stdout=log,
                                stderr=subprocess.STDOUT, env=dict(env, CUDA_VISIBLE_DEVICES="0")).returncode
        print(f"==== {name} exit {rc} ({hours()})\n" + "\n".join((LOGS / f"{name}.log").read_text(errors="replace").splitlines()[-12:]), flush=True)
        check = json.loads((WORK / name / "check.json").read_text()) if (WORK / name / "check.json").exists() else {}
        if rc == 0 and check.get("ok"):
            if check["tri"] < int(args[args.index("--tri") + 1]):   # smaller micro-batches, same optimizer batch
                f = int(args[args.index("--tri") + 1]) // check["tri"]
                args[args.index("--tri") + 1] = str(check["tri"])
                args[args.index("--accum") + 1] = str(int(args[args.index("--accum") + 1]) * f)
            chosen = (b, path, args); break
    assert chosen, "no base passed the fp16 / memory check"
    b, path, args = chosen
    print(f"training {b['tag']} from {path} for at most {CONFIG['hours']} h", flush=True)
    log = open(LOGS / f"{b['tag']}.log", "w")
    p = subprocess.Popen([sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", "2",
                          str(WORK / "l24_worker.py"), "--stage", "1",
                          "--out", str(WORK / b["tag"]), "--hours", str(CONFIG["hours"]), "--class-weights", *args],
                         stdout=log, stderr=subprocess.STDOUT, env=env)
    rc = follow([(b["tag"], p)])
    log.close()
    meta = json.loads((WORK / b["tag"] / "meta.json").read_text())
    assert rc == [0] and (WORK / b["tag"] / "adapter_model.safetensors").exists(), "stage 1 failed"
    print("STAGE 1 DONE", b["tag"], json.dumps({k: meta[k] for k in ("steps", "total_steps", "planned_steps", "hours",
          "epochs_done", "epochs_total", "final_loss", "overflow_microbatches", "skipped_steps")}), hours(), flush=True)

if CONFIG["stage"] == 2:
    s1 = sorted({os.path.dirname(m) for m in glob.glob(f"/kaggle/input/**/{CONFIG['tag']}/meta.json", recursive=True)
                 if json.load(open(m)).get("stage") == 1})
    assert len(s1) == 1, s1
    s1 = s1[0]
    s1_meta = json.load(open(os.path.join(s1, "meta.json")))
    print("stage 1:", s1, json.dumps({k: s1_meta.get(k) for k in ("base", "steps", "hours", "epochs_done", "final_loss")}), flush=True)
    path = s1_meta["base"]   # the base stage 1 chose, mounted at the same path here
    assert os.path.exists(os.path.join(path, "config.json")), path
    train["label"] = train.model.map({l: i for i, l in enumerate(CONFIG["labels"])})
    train["fold"] = [int(c) for c in CONFIG["folds"]]
    assert train.label.notna().all() and train.groupby("prompt_id").fold.nunique().max() == 1
    train[["id", "text", "label", "fold", "prompt_id"]].to_parquet(TMP / "train.parquet", index=False)
    procs = []
    for gpu, jobs in enumerate(CONFIG["gpu_jobs"]):
        if gpu:
            time.sleep(60)
        name = f"stage2_gpu{gpu}"
        command = worker + ["--stage", "2", "--base", path, "--labels", labels, "--train", str(TMP / "train.parquet"),
                            "--init", s1, "--jobs", jobs, "--out", str(WORK / CONFIG["tag"]), *CONFIG["args"]]
        print("launch", name, jobs, " ".join(CONFIG["args"]), flush=True)
        procs.append((name, subprocess.Popen(command, stdout=open(LOGS / f"{name}.log", "w"), stderr=subprocess.STDOUT,
                                             env=dict(env, CUDA_VISIBLE_DEVICES=str(gpu)))))
    rcs = follow(procs, every=300)
    folds = train.fold.values
    oof = np.zeros((len(train), len(CONFIG["labels"])), np.float32); done = []
    for k in range(5):
        f = WORK / f"{CONFIG['tag']}_f{k}" / "valid.npz"
        if f.exists():
            z = np.load(f)
            assert list(z["ids"]) == list(train.id.values[folds == k])
            oof[folds == k] = z["logits"]; done.append(k)
    summary = {"stage1": {k: s1_meta.get(k) for k in ("base", "steps", "hours", "epochs_done", "final_loss")}, "folds_done": done}
    for d in sorted(WORK.glob(f"{CONFIG['tag']}*/meta.json")):
        mm = json.loads(d.read_text())
        summary[d.parent.name] = {k: mm.get(k) for k in ("job", "steps", "minutes", "final_loss", "in_sample_check_40",
                                                           "valid_accuracy", "valid_balanced_accuracy", "overflow_microbatches")}
    if len(done) == 5:
        y = train.label.astype(int).values
        summary["oof_balanced_accuracy"] = float(np.mean([(oof[y == c].argmax(1) == c).mean() for c in range(len(CONFIG["labels"]))]))
        out = WORK / "enc" / CONFIG["tag"]
        out.mkdir(parents=True, exist_ok=True)
        np.savez(out / "oof.npz", ids=np.array(train.id.tolist()), labels=np.array(CONFIG["labels"]), logits=oof)
    full = WORK / CONFIG["tag"]
    if (full / "heads.pt").exists():
        # reload check: infer.py's merge path (nlm_oof.py) on the rows the worker checked in-process
        sys.path.insert(0, str(WORK))
        import nlm_oof
        spec = dict(name=CONFIG["tag"], path=path, keep=s1_meta["keep"], layers=s1_meta["layers"], bs=8, max_len=s1_meta["max_len"])
        tok, m, mods, first = nlm_oof.load_base(spec)
        with nlm_oof.merged(mods, str(full)) as H:
            feats = nlm_oof.pooled(spec, tok, m, first, train.text.values[:40], spec["layers"], [0])
            lp = nlm_oof.head_lp(H, feats, spec["layers"])
        ref = np.load(full / "check40.npy")
        summary["reload_check"] = dict(max_abs_diff=float(np.abs(lp - ref).max()), argmax_agreement=float((lp.argmax(1) == ref.argmax(1)).mean()),
                                       accuracy_40=float((lp.argmax(1) == train.label.astype(int).values[:40]).mean()))
        print("reload check", summary["reload_check"], flush=True)
    (WORK / "l24_summary.json").write_text(json.dumps(summary, indent=1))
    print("STAGE 2 DONE", json.dumps(summary, indent=1), hours(), flush=True)
    assert rcs == [0, 0] and len(done) == 5 and summary.get("reload_check", {}).get("argmax_agreement", 0) >= 0.9, "stage 2 incomplete"